# CSC 6302 Practical 4 — Deep Learning for Vision\n**Student:** Munyeme Sinyani (SINYANI) · **2025024319** · University of Zambia\n\nRuntime → Change runtime type → **T4 GPU**.\n\nTasks: TinyCNN on CIFAR-10 → transfer learning on my bird dataset → Grad-CAM → Gradio deploy notes.

In [ ]:
!pip -q install gradio torch torchvision matplotlib scikit-learn pillow\nimport torch, torchvision\nprint('cuda', torch.cuda.is_available(), torch.__version__)

## Task 1 — TinyCNN from scratch on CIFAR-10

In [ ]:
import time\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Subset\nfrom torchvision import datasets, transforms\n\nDEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'\nEPOCHS = 12\n\nclass TinyCNN(nn.Module):\n    def __init__(self, n_classes=10):\n        super().__init__()\n        self.features = nn.Sequential(\n            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),\n            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),\n            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),\n        )\n        self.classifier = nn.Sequential(\n            nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(inplace=True), nn.Dropout(0.4), nn.Linear(256, n_classes)\n        )\n    def forward(self, x):\n        return self.classifier(self.features(x))\n\ntf_train = transforms.Compose([\n    transforms.RandomHorizontalFlip(), transforms.RandomCrop(32, padding=4), transforms.ToTensor(),\n    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))])\ntf_test = transforms.Compose([transforms.ToTensor(),\n    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))])\n\ntrain_full = datasets.CIFAR10('./cifar10', train=True, download=True, transform=tf_train)\ntest_set = datasets.CIFAR10('./cifar10', train=False, download=True, transform=tf_test)\nidx = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(0)).tolist()\nn_val = len(train_full)//10\nval_base = datasets.CIFAR10('./cifar10', train=True, download=False, transform=tf_test)\ntrain_loader = DataLoader(Subset(train_full, idx[n_val:]), batch_size=128, shuffle=True, num_workers=2)\nval_loader = DataLoader(Subset(val_base, idx[:n_val]), batch_size=128, num_workers=2)\ntest_loader = DataLoader(test_set, batch_size=128, num_workers=2)\n\nmodel = TinyCNN().to(DEVICE)\nopt = torch.optim.Adam(model.parameters(), lr=1e-3)\nhist = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc']}\n\ndef run_epoch(loader, train=True):\n    model.train(train)\n    loss_sum=correct=total=0\n    for x,y in loader:\n        x,y = x.to(DEVICE), y.to(DEVICE)\n        if train: opt.zero_grad()\n        logits = model(x); loss = F.cross_entropy(logits, y)\n        if train: loss.backward(); opt.step()\n        loss_sum += loss.item()*x.size(0)\n        correct += (logits.argmax(1)==y).sum().item(); total += x.size(0)\n    return loss_sum/total, correct/total\n\nfor ep in range(EPOCHS):\n    tr = run_epoch(train_loader, True); va = run_epoch(val_loader, False)\n    for k,v in zip(hist, tr+va): hist[k].append(v)\n    print(f'epoch {ep+1}/{EPOCHS} train_acc={tr[1]:.3f} val_acc={va[1]:.3f}')\n\nfig, ax = plt.subplots(1,2, figsize=(9,3))\nax[0].plot(hist['train_loss'], label='train'); ax[0].plot(hist['val_loss'], label='val'); ax[0].legend(); ax[0].set_title('Loss')\nax[1].plot(hist['train_acc'], label='train'); ax[1].plot(hist['val_acc'], label='val'); ax[1].legend(); ax[1].set_title('Accuracy')\nplt.show()\n\ncm = np.zeros((10,10), int); correct=total=0\nmodel.eval()\nwith torch.no_grad():\n    for x,y in test_loader:\n        pred = model(x.to(DEVICE)).argmax(1).cpu()\n        for yt,yp in zip(y, pred): cm[int(yt), int(yp)] += 1\n        correct += (pred==y).sum().item(); total += y.size(0)\nprint('test accuracy', correct/total)\nplt.figure(figsize=(6,5)); plt.imshow(cm, cmap='Blues'); plt.xticks(range(10), test_set.classes, rotation=45, ha='right'); plt.yticks(range(10), test_set.classes); plt.title('Confusion'); plt.colorbar(); plt.tight_layout(); plt.show()\nprint('Discussion: overfitting starts when train accuracy keeps rising while val stalls/diverges.')\nprint('Common confusions: cat↔dog, automobile↔truck — similar shapes/colours in 32×32 CIFAR.')

## Task 2–3 — Transfer learning + Grad-CAM on my bird dataset\nUpload a zip of `images/bird_dataset` (train/val/test) to Colab, or mount Drive. Local laptop already runs `prac4/train_prac4.py`.

In [ ]:
# Optional: upload bird_dataset.zip then:\n# !unzip -q bird_dataset.zip -d bird_dataset\nprint('On the laptop I use: python prac4/prepare_bird_dataset.py && python prac4/train_prac4.py')\nprint('That trains TinyCNN scratch vs MobileNet frozen vs fine-tuned, saves model.pt + Grad-CAM.')

## Task 4 — Hugging Face deploy\nApp: `prac4/app.py` (top-3 `gr.Label`). Model repo: https://huggingface.co/Salaam21/csc6302-prac4-SINYANI\n\n**Restriction:** creating a Gradio Space on free CPU returned 402 — HF requires PRO for Gradio Spaces. App + weights are on the model repo; local `python prac4/app.py` works.

## AI disclosure\nComposer helped with notebook layout and Gradio wiring. Training choices, dataset documentation, and write-ups are mine.